# 07 — Conversation Memory

Rolling-window history injected into every prompt.

| Parameter | Value | Meaning |
|---|---|---|
| `window_size` | 6 | Keep last 6 messages |
| `get_context(last_n=4)` | 4 | Inject last 4 into prompt |

**Config source:** `configs/default.yaml` -> `paths.faiss_index`, `llm`

In [ ]:
import sys
import os
sys.path.append("..")

from rag_pipeline.utils import load_notebook_config, format_docs
from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.vectorstores import load_vectorstore
from rag_pipeline.retrieval import build_retriever
from rag_pipeline.generation import build_llms
from rag_pipeline.generation.prompts import CONVERSATION_PROMPT
from rag_pipeline.memory import ConversationMemory

cfg, REPO = load_notebook_config()
FAISS_DIR = REPO / cfg.paths["faiss_index"]

**Build store, retriever, LLM**

In [ ]:
emb = build_embeddings(dict(cfg.embeddings))
store = load_vectorstore(emb, {"type": "faiss", "persist_dir": str(FAISS_DIR)})
retriever = build_retriever(store, {"search_type": "similarity", "k": 2})

llms = build_llms(dict(cfg.llm))
model_name = next(iter(llms))
llm = llms[model_name]
print("Using model:", model_name)

**Conversation loop**

In [ ]:
memory = ConversationMemory(window_size=6)

conversation = [
    "What are large language models?",
    "What are architectures used in large language models?",
    "What is LangChain?",
]

for q in conversation:
    memory.add_message("user", q)
    ctx = format_docs(retriever.invoke(q), max_chars=120)
    prompt = CONVERSATION_PROMPT.format(
        conversation_context=memory.get_context(),
        context=ctx,
        question=q,
    )
    print(f"\n{'─'*60}\nUSER: {q}\n{'─'*60}")
    out = "".join(str(c) for c in llm.stream(prompt, max_new_tokens=150))
    print(out)
    memory.add_message("assistant", out[:300])

**Memory state**

In [ ]:
print("\nMemory stats:", memory.get_stats())
for role, content in memory.messages:
    print(f"{role.upper()}: {content[:120]}...")